
# Tech Challenge FIAP — Fase 3
## Geração de Dados Hospitalares Sintéticos

### Objetivo

Criar uma base de dados fictícia para simular o ambiente de um hospital e complementar o dataset MedQuAD.

Os dados serão utilizados em três componentes:

1. **Fine-tuning:** perguntas e respostas sobre protocolos e regras institucionais.
2. **RAG:** recuperação de documentos e protocolos internos.
3. **Banco de dados:** consultas a pacientes, exames e registros estruturados.

### Hospital fictício

**Nome:** Hospital Aurora

**Cenário:** Assistente de apoio à decisão clínica, destinado ao uso supervisionado por profissionais de saúde.

### Segurança

Todos os pacientes e registros serão sintéticos.

Os protocolos iniciais representam fluxos operacionais de demonstração, não diretrizes médicas validadas. Recomendações clínicas específicas deverão ser fundamentadas em referências confiáveis e submetidas à revisão profissional.

O assistente não terá autonomia para realizar prescrições ou tomar decisões clínicas definitivas.


In [2]:
from google.colab import drive
from pathlib import Path
import json
from textwrap import dedent

drive.mount("/content/drive")

base_dir = Path(
    "/content/drive/MyDrive/FIAP/TechChallenge_Fase3/data"
)

protocols_dir = base_dir / "protocols"
synthetic_dir = base_dir / "synthetic"

protocols_dir.mkdir(parents=True, exist_ok=True)
synthetic_dir.mkdir(parents=True, exist_ok=True)

print("Diretórios configurados.")

Mounted at /content/drive
Diretórios configurados.



## 1. Protocolos hospitalares

Os protocolos definem procedimentos e regras que deverão orientar o assistente.

Cada documento terá:

- Identificador e título.
- Versão do documento.
- Objetivo.
- Procedimentos ou orientações.
- Restrições de segurança.

Esses documentos serão utilizados posteriormente pelo RAG, permitindo que a LLM fundamente suas respostas em informações recuperadas da base documental.


In [3]:
protocols = {
    "PRT-001": dedent("""
        # PRT-001 — Verificação de Exames Pendentes

        Versão: 1.0
        Hospital: Aurora
        Tipo: Protocolo operacional sintético

        ## Objetivo
        Identificar exames solicitados que ainda não
        possuem resultados cadastrados.

        ## Procedimento
        1. Identificar o paciente pelo ID.
        2. Consultar os exames registrados.
        3. Identificar exames com status "pendente".
        4. Informar as pendências ao profissional.
        5. Registrar alertas para acompanhamento.

        ## Restrições
        - Não interpretar resultados inexistentes.
        - Não presumir que todo exame pendente está atrasado.
        - Não prescrever tratamentos.
        - Encaminhar decisões clínicas para revisão humana.
    """).strip(),

    "PRT-002": dedent("""
        # PRT-002 — Validação de Recomendações Clínicas

        Versão: 1.0
        Hospital: Aurora
        Tipo: Protocolo operacional sintético

        ## Objetivo
        Garantir que sugestões produzidas pelo assistente
        sejam fundamentadas e avaliadas por profissionais.

        ## Procedimento
        1. Consultar os protocolos aplicáveis.
        2. Identificar informações relevantes do paciente.
        3. Elaborar uma resposta fundamentada.
        4. Apresentar as fontes utilizadas.
        5. Encaminhar sugestões clínicas para revisão humana.

        ## Restrições
        - Não emitir prescrições automaticamente.
        - Não confirmar diagnósticos de forma autônoma.
        - Não apresentar informações sem fundamento
          como se fossem recomendações oficiais.
        - Informar quando houver evidências insuficientes.
    """).strip()
}

for protocol_id, content in protocols.items():
    path = protocols_dir / f"{protocol_id}.md"
    path.write_text(content, encoding="utf-8")

print(f"Protocolos criados: {len(protocols)}")

Protocolos criados: 2



## 2. Base de pacientes sintéticos

Serão criados registros fictícios contendo:

- Identificador do paciente.
- Idade.
- Condições clínicas cadastradas.
- Exames solicitados.
- Status dos exames.

Os dados serão utilizados para testar consultas estruturadas e respostas contextualizadas.

Nenhuma informação corresponde a pacientes reais.


In [4]:
patients = [
    {
        "patient_id": "P001",
        "age": 58,
        "conditions": [
            "Diabetes mellitus tipo 2",
            "Hipertensão arterial"
        ],
        "exams": [
            {
                "exam_id": "E001",
                "name": "Hemoglobina glicada",
                "status": "pendente"
            },
            {
                "exam_id": "E002",
                "name": "Creatinina",
                "status": "concluido"
            }
        ]
    },
    {
        "patient_id": "P002",
        "age": 45,
        "conditions": [
            "Hipertensão arterial"
        ],
        "exams": [
            {
                "exam_id": "E003",
                "name": "Creatinina",
                "status": "pendente"
            },
            {
                "exam_id": "E004",
                "name": "Potássio",
                "status": "pendente"
            }
        ]
    },
    {
        "patient_id": "P003",
        "age": 37,
        "conditions": [
            "Diabetes mellitus tipo 2"
        ],
        "exams": [
            {
                "exam_id": "E005",
                "name": "Hemoglobina glicada",
                "status": "concluido"
            }
        ]
    }
]

output_path = synthetic_dir / "patients.json"

output_path.write_text(
    json.dumps(patients, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(f"Pacientes criados: {len(patients)}")

Pacientes criados: 3



## 3. Perguntas e respostas institucionais

Serão criados exemplos sintéticos de perguntas e respostas baseados nos protocolos do Hospital Aurora.

O objetivo é complementar o MedQuAD com comportamentos específicos do assistente, incluindo:

- Consulta de exames pendentes.
- Identificação de limitações.
- Fundamentação das respostas.
- Encaminhamento para revisão humana.

Esses exemplos serão revisados antes da inclusão no fine-tuning.

Dados individuais de pacientes não serão utilizados no treinamento.


In [5]:
hospital_qa = [
    {
        "question": "Como identificar exames pendentes?",
        "answer": (
            "O assistente deve consultar os exames do paciente, "
            "identificar os registros com status pendente e "
            "informar as pendências ao profissional responsável."
        ),
        "source": "HospitalAurora",
        "protocol_id": "PRT-001"
    },
    {
        "question": "Um exame pendente significa que está atrasado?",
        "answer": (
            "Não necessariamente. O status pendente indica "
            "que ainda não existe resultado cadastrado. "
            "É necessário consultar os prazos para identificar atrasos."
        ),
        "source": "HospitalAurora",
        "protocol_id": "PRT-001"
    },
    {
        "question": "O assistente pode prescrever medicamentos?",
        "answer": (
            "Não. O assistente atua como ferramenta de apoio "
            "e não realiza prescrições automaticamente. "
            "As decisões clínicas devem ser tomadas "
            "por profissionais habilitados."
        ),
        "source": "HospitalAurora",
        "protocol_id": "PRT-002"
    },
    {
        "question": "Como o assistente deve fundamentar uma resposta?",
        "answer": (
            "O assistente deve consultar os protocolos disponíveis, "
            "utilizar informações relevantes e apresentar "
            "as fontes que sustentam a resposta."
        ),
        "source": "HospitalAurora",
        "protocol_id": "PRT-002"
    },
    {
        "question": "O que fazer quando não existem evidências suficientes?",
        "answer": (
            "O assistente deve informar a insuficiência "
            "de evidências e encaminhar a questão "
            "para avaliação profissional."
        ),
        "source": "HospitalAurora",
        "protocol_id": "PRT-002"
    }
]

output_path = synthetic_dir / "hospital_qa.jsonl"

with output_path.open("w", encoding="utf-8") as file:
    for record in hospital_qa:
        file.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

print(f"Perguntas e respostas criadas: {len(hospital_qa)}")

Perguntas e respostas criadas: 5


In [6]:
# Validar identificadores de pacientes e exames
patient_ids = [p["patient_id"] for p in patients]

exam_ids = [
    exam["exam_id"]
    for patient in patients
    for exam in patient["exams"]
]

assert len(patient_ids) == len(set(patient_ids))
assert len(exam_ids) == len(set(exam_ids))

# Validar status dos exames
valid_statuses = {"pendente", "concluido"}

for patient in patients:
    for exam in patient["exams"]:
        assert exam["status"] in valid_statuses

# Validar referências aos protocolos
for record in hospital_qa:
    assert record["protocol_id"] in protocols

# Validar arquivos
assert (synthetic_dir / "patients.json").exists()
assert (synthetic_dir / "hospital_qa.jsonl").exists()

print("Validação concluída com sucesso!")
print(f"Pacientes: {len(patients)}")
print(f"Exames: {len(exam_ids)}")
print(f"Protocolos: {len(protocols)}")
print(f"Perguntas e respostas: {len(hospital_qa)}")

Validação concluída com sucesso!
Pacientes: 3
Exames: 5
Protocolos: 2
Perguntas e respostas: 5



## 4. Modelos de documentos hospitalares

Para complementar os dados institucionais, serão criados três modelos de documentos sintéticos:

- Laudo de exame.
- Receita médica.
- Registro de procedimento.

Os modelos representam estruturas documentais fictícias e não contêm recomendações clínicas ou prescrições reais.

Todos os documentos produzidos pelo assistente deverão permanecer como rascunhos até a validação de um profissional habilitado.


In [7]:
document_templates = {
    "MOD-001": {
        "type": "laudo",
        "title": "Modelo de Laudo de Exame",
        "fields": [
            "patient_id",
            "exam_name",
            "exam_date",
            "findings",
            "conclusion",
            "responsible_professional"
        ],
        "requires_approval": True
    },
    "MOD-002": {
        "type": "receita",
        "title": "Modelo de Receita Médica",
        "fields": [
            "patient_id",
            "medication",
            "dosage",
            "instructions",
            "prescriber",
            "signature"
        ],
        "requires_approval": True
    },
    "MOD-003": {
        "type": "procedimento",
        "title": "Registro de Procedimento",
        "fields": [
            "patient_id",
            "procedure",
            "date",
            "description",
            "responsible_professional"
        ],
        "requires_approval": True
    }
}

output_path = synthetic_dir / "document_templates.json"

output_path.write_text(
    json.dumps(
        document_templates,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

assert all(
    template["requires_approval"]
    for template in document_templates.values()
)

print(f"Modelos criados: {len(document_templates)}")

Modelos criados: 3



## 5. Expansão das perguntas e respostas institucionais

O conjunto inicial de cinco perguntas e respostas será ampliado com exemplos adicionais sobre os procedimentos e as regras de segurança do Hospital Aurora.

### Objetivos

- Representar diferentes formas de consultar os protocolos internos.
- Ensinar ao modelo os limites operacionais do assistente.
- Reforçar a necessidade de validação humana.
- Incluir exemplos sobre documentos hospitalares.
- Evitar o uso de dados individuais de pacientes no fine-tuning.

Os novos exemplos serão construídos a partir dos protocolos sintéticos existentes.

Essa expansão representa uma etapa inicial de personalização institucional. Posteriormente, serão incorporados exemplos de protocolos clínicos fundamentados em referências médicas.


In [9]:
import json
from pathlib import Path

synthetic_dir = (
    base_dir / "synthetic"
)

qa_path = synthetic_dir / "hospital_qa.jsonl"

# Recuperar exemplos existentes
with qa_path.open("r", encoding="utf-8") as file:
    hospital_qa = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

print(f"Exemplos existentes: {len(hospital_qa)}")

new_examples = [
    (
        "Qual é o primeiro passo para consultar exames pendentes?",
        "Identificar o paciente pelo seu ID antes de consultar os registros.",
        "PRT-001"
    ),
    (
        "Como identificar um exame pendente?",
        "Consultar os registros do paciente e verificar os exames com status pendente.",
        "PRT-001"
    ),
    (
        "O assistente pode interpretar um exame sem resultado?",
        "Não. O assistente não deve interpretar resultados inexistentes.",
        "PRT-001"
    ),
    (
        "Como informar exames pendentes ao médico?",
        "Apresentar os exames identificados como pendentes e indicar que ainda não possuem resultados cadastrados.",
        "PRT-001"
    ),
    (
        "O assistente pode afirmar que um exame pendente está atrasado?",
        "Não. O status pendente não comprova atraso. É necessário verificar os prazos aplicáveis.",
        "PRT-001"
    ),
    (
        "O que fazer quando um exame pendente é identificado?",
        "Informar a pendência ao profissional responsável e registrar o alerta para acompanhamento.",
        "PRT-001"
    ),
    (
        "O assistente pode prescrever tratamento a partir de exames pendentes?",
        "Não. A identificação de pendências não autoriza prescrições ou decisões clínicas autônomas.",
        "PRT-001"
    ),
    (
        "Como uma sugestão clínica deve ser validada?",
        "A sugestão deve ser fundamentada nos protocolos aplicáveis e encaminhada para avaliação de um profissional habilitado.",
        "PRT-002"
    ),
    (
        "O assistente pode confirmar um diagnóstico?",
        "Não. O assistente não deve confirmar diagnósticos de forma autônoma.",
        "PRT-002"
    ),
    (
        "O que fazer quando não há protocolo aplicável?",
        "Informar que não foi encontrada fundamentação institucional suficiente e encaminhar a questão para avaliação profissional.",
        "PRT-002"
    ),
    (
        "Uma resposta clínica deve apresentar suas fontes?",
        "Sim. As fontes utilizadas devem acompanhar a resposta para permitir sua verificação.",
        "PRT-002"
    ),
    (
        "O assistente pode emitir uma recomendação clínica definitiva?",
        "Não. As sugestões clínicas devem permanecer sujeitas à validação de um profissional habilitado.",
        "PRT-002"
    ),
    (
        "Um laudo gerado pelo assistente é definitivo?",
        "Não. Um laudo gerado automaticamente deve ser tratado como rascunho até sua validação pelo profissional responsável.",
        "PRT-002"
    ),
    (
        "O assistente pode assinar uma receita médica?",
        "Não. O assistente não possui autoridade para assinar ou emitir prescrições médicas.",
        "PRT-002"
    ),
    (
        "Como utilizar o modelo de registro de procedimento?",
        "O modelo deve ser preenchido com informações verificadas e submetido à aprovação do profissional responsável.",
        "PRT-002"
    ),
    (
        "O assistente pode apresentar informações sem fundamento como recomendações oficiais?",
        "Não. Informações sem fundamento identificado não devem ser apresentadas como recomendações oficiais do hospital.",
        "PRT-002"
    )
]

existing_questions = {
    item["question"].strip().casefold()
    for item in hospital_qa
}

for question, answer, protocol_id in new_examples:
    key = question.strip().casefold()

    if key not in existing_questions:
        hospital_qa.append({
            "question": question,
            "answer": answer,
            "source": "HospitalAurora",
            "protocol_id": protocol_id
        })

        existing_questions.add(key)

# Persistir o conjunto atualizado
with qa_path.open("w", encoding="utf-8") as file:
    for record in hospital_qa:
        file.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

print(f"Total de exemplos: {len(hospital_qa)}")

Exemplos existentes: 5
Total de exemplos: 21


In [10]:
# Verificar campos obrigatórios
required_fields = {
    "question",
    "answer",
    "source",
    "protocol_id"
}

for item in hospital_qa:
    assert required_fields.issubset(item.keys())

    assert item["question"].strip()
    assert item["answer"].strip()

    assert item["protocol_id"] in {
        "PRT-001",
        "PRT-002"
    }

# Verificar duplicações de perguntas
questions = [
    item["question"].strip().casefold()
    for item in hospital_qa
]

assert len(questions) == len(set(questions))

print("Dataset institucional validado!")
print(f"Total de perguntas e respostas: {len(hospital_qa)}")

Dataset institucional validado!
Total de perguntas e respostas: 21



## 6. Protocolo clínico — Acompanhamento do Diabetes Tipo 2

Será criado o protocolo PRT-003, voltado à identificação de informações relevantes no acompanhamento de pacientes com diabetes mellitus tipo 2.

O documento será fundamentado nas diretrizes de 2026 da American Diabetes Association (ADA) e da Sociedade Brasileira de Diabetes (SBD).

### Objetivos

- Orientar a consulta de registros de hemoglobina glicada.
- Identificar informações relacionadas ao acompanhamento da função renal.
- Diferenciar exames pendentes de exames não cadastrados.
- Fornecer referências para as respostas do assistente.
- Garantir que decisões clínicas permaneçam sob responsabilidade profissional.

O protocolo é sintético, desenvolvido para demonstração acadêmica, e não representa um documento aprovado por um hospital real.


In [11]:
from google.colab import drive
from pathlib import Path
from textwrap import dedent

drive.mount("/content/drive")

base_dir = Path(
    "/content/drive/MyDrive/FIAP/TechChallenge_Fase3"
)

protocols_dir = base_dir / "data/protocols"
protocols_dir.mkdir(parents=True, exist_ok=True)

protocol = dedent("""
    # PRT-003 — Acompanhamento do Diabetes Tipo 2

    **Hospital:** Aurora
    **Versão:** 1.0
    **Tipo:** Protocolo clínico sintético
    **Status:** Demonstração acadêmica; não validado para uso clínico

    ## 1. Objetivo

    Apoiar a revisão de informações clínicas e exames
    registrados de pacientes com diabetes mellitus tipo 2.

    ## 2. Monitoramento glicêmico

    A hemoglobina glicada (HbA1c) é utilizada no
    acompanhamento do controle glicêmico.

    A ADA 2026 orienta a avaliação do controle glicêmico
    pelo menos duas vezes ao ano, podendo ser mais
    frequente, por exemplo, a cada três meses quando
    as metas não são atingidas ou há mudanças terapêuticas.

    A frequência individual deve ser definida pelo
    profissional responsável.

    ## 3. Avaliação da função renal

    Em pacientes com DM2, recomenda-se avaliar
    periodicamente:

    - Relação albumina/creatinina urinária (RAC).
    - Taxa de filtração glomerular estimada (TFGe).

    A ADA 2026 recomenda avaliação pelo menos anual,
    com acompanhamento individualizado quando
    houver doença renal.

    ## 4. Verificação de registros

    O assistente deverá:

    1. Identificar o paciente pelo ID.
    2. Consultar as condições clínicas cadastradas.
    3. Consultar os exames e seus respectivos status.
    4. Apresentar exames pendentes.
    5. Indicar informações relevantes não cadastradas.
    6. Apresentar as referências utilizadas.

    Um exame não cadastrado não deve ser classificado
    automaticamente como pendente ou atrasado.

    ## 5. Limites de atuação

    O assistente não poderá:

    - Confirmar diagnósticos autonomamente.
    - Prescrever ou modificar tratamentos.
    - Interpretar resultados que não estejam disponíveis.
    - Determinar automaticamente que exames estão atrasados.
    - Substituir a avaliação do profissional responsável.

    ## 6. Referências

    ADA — Standards of Care in Diabetes 2026, seção 6:
    https://doi.org/10.2337/dc26-s006

    ADA — Standards of Care in Diabetes 2026, seção 11:
    https://doi.org/10.2337/dc26-s011

    Sociedade Brasileira de Diabetes — Diretrizes 2026:
    https://diretriz.diabetes.org.br/
""").strip()

protocol_path = protocols_dir / "PRT-003.md"

protocol_path.write_text(
    protocol,
    encoding="utf-8"
)

print(f"Protocolo criado: {protocol_path.name}")
print(f"Caracteres: {len(protocol)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Protocolo criado: PRT-003.md
Caracteres: 1987



## 7. Perguntas e respostas do protocolo PRT-003

Serão adicionados exemplos supervisionados relacionados ao acompanhamento do diabetes mellitus tipo 2.

As perguntas abordarão monitoramento glicêmico, avaliação renal, interpretação dos status dos exames e limites de atuação do assistente.

As respostas serão fundamentadas no protocolo PRT-003.

O conteúdo será utilizado para complementar o conjunto de treinamento institucional, preservando as referências aos documentos de origem.


In [12]:
import json

synthetic_dir = base_dir / "data/synthetic"
qa_path = synthetic_dir / "hospital_qa.jsonl"

with qa_path.open("r", encoding="utf-8") as file:
    hospital_qa = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

new_examples = [
    (
        "Qual exame pode ser utilizado no acompanhamento glicêmico do DM2?",
        "A hemoglobina glicada (HbA1c) é um dos exames utilizados "
        "para acompanhar o controle glicêmico."
    ),
    (
        "Com qual frequência a HbA1c deve ser avaliada?",
        "Segundo a ADA 2026, o controle glicêmico deve ser avaliado "
        "pelo menos duas vezes ao ano, com maior frequência "
        "conforme a situação clínica."
    ),
    (
        "Quando pode ser necessária uma avaliação glicêmica mais frequente?",
        "Quando as metas glicêmicas não são atingidas, há mudanças "
        "no tratamento ou outras condições que exigem acompanhamento "
        "mais próximo, conforme avaliação profissional."
    ),
    (
        "Quais avaliações são relevantes para a função renal no DM2?",
        "A avaliação inclui a relação albumina/creatinina urinária "
        "(RAC) e a taxa de filtração glomerular estimada (TFGe)."
    ),
    (
        "Qual é a frequência mínima recomendada para avaliação renal no DM2?",
        "A ADA 2026 recomenda avaliar RAC e TFGe pelo menos "
        "anualmente, com frequência individualizada em "
        "pacientes com doença renal."
    ),
    (
        "O que fazer quando não há data registrada para um exame?",
        "O assistente deve informar a ausência da data e não "
        "classificar automaticamente o exame como atrasado."
    ),
    (
        "Um exame não cadastrado deve ser considerado pendente?",
        "Não. Um exame sem registro deve ser identificado como "
        "não cadastrado. O status pendente depende de um "
        "registro existente com essa classificação."
    ),
    (
        "O assistente pode modificar o tratamento de um paciente com DM2?",
        "Não. O assistente não tem autonomia para prescrever "
        "ou alterar tratamentos. Essas decisões pertencem "
        "ao profissional responsável."
    ),
    (
        "O assistente pode interpretar um resultado de HbA1c inexistente?",
        "Não. Sem o resultado disponível, o assistente deve "
        "informar a ausência dos dados e evitar interpretações."
    ),
    (
        "Como o assistente deve apresentar exames pendentes?",
        "Deve identificar o paciente, consultar os registros "
        "e apresentar os exames com status pendente, "
        "sem presumir resultados ou atrasos."
    ),
    (
        "A creatinina cadastrada garante que a TFGe está disponível?",
        "Não. O assistente deve verificar se a TFGe foi "
        "calculada ou registrada, sem presumir sua disponibilidade."
    ),
    (
        "O assistente pode confirmar doença renal a partir de exames isolados?",
        "Não. O assistente deve apresentar os dados disponíveis "
        "e encaminhar a interpretação e o diagnóstico "
        "ao profissional habilitado."
    )
]

existing_questions = {
    item["question"].strip().casefold()
    for item in hospital_qa
}

added = 0

for question, answer in new_examples:
    key = question.strip().casefold()

    if key not in existing_questions:
        hospital_qa.append({
            "question": question,
            "answer": answer,
            "source": "HospitalAurora",
            "protocol_id": "PRT-003"
        })

        existing_questions.add(key)
        added += 1

with qa_path.open("w", encoding="utf-8") as file:
    for item in hospital_qa:
        file.write(
            json.dumps(item, ensure_ascii=False) + "\n"
        )

print(f"Exemplos adicionados: {added}")
print(f"Total de exemplos: {len(hospital_qa)}")

Exemplos adicionados: 12
Total de exemplos: 33


In [13]:
protocol_ids = {
    path.stem
    for path in protocols_dir.glob("PRT-*.md")
}

questions = [
    item["question"].strip().casefold()
    for item in hospital_qa
]

assert len(questions) == len(set(questions))

assert all(
    item["protocol_id"] in protocol_ids
    for item in hospital_qa
)

assert all(
    item["question"].strip() and item["answer"].strip()
    for item in hospital_qa
)

print("Validação concluída!")
print(f"Protocolos: {len(protocol_ids)}")
print(f"Perguntas e respostas: {len(hospital_qa)}")

Validação concluída!
Protocolos: 3
Perguntas e respostas: 33
